# Multi-Label Clause Classification Dataset Pipeline

**Purpose**: Transform the CUAD dataset into a proper multi-label format (41 labels)
for Legal-BERT fine-tuning with contract-level provenance tracking.

**Output Format**: `| text | contract_id | label_1 | label_2 | ... | label_41 |`

This notebook is the foundation for the final transformer-based legal clause classifier.

---
## Section 1 — Imports & Configuration

In [1]:
import json
import random
import hashlib
import os
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================

RANDOM_SEED = 42
CONTEXT_WINDOW = 250       # chars of surrounding context on each side
NEGATIVE_SNIPPET_LEN = 500 # target length for negative examples

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Paths
BASE_DIR = r"C:\Users\chari\Desktop\Contract_Intelligence_AI"
CUAD_PATH = os.path.join(BASE_DIR, "data", "raw", "dataset", "CUAD_v1", "CUAD_v1.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "processed")

CSV_OUTPUT = os.path.join(OUTPUT_DIR, "multi_label_clause_dataset.csv")
LABEL_MAP_OUTPUT = os.path.join(OUTPUT_DIR, "label_mapping.json")
STATS_OUTPUT = os.path.join(OUTPUT_DIR, "dataset_statistics.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration ready.")
print(f"  CUAD path : {CUAD_PATH}")
print(f"  Output dir: {OUTPUT_DIR}")

Configuration ready.
  CUAD path : C:\Users\chari\Desktop\Contract_Intelligence_AI\data\raw\dataset\CUAD_v1\CUAD_v1.json
  Output dir: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed


---
## Section 2 — Load CUAD Dataset

In [2]:
with open(CUAD_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

contracts = data["data"]

print("Dataset Loaded Successfully")
print(f"Total Contracts: {len(contracts)}")
print(f"Sample contract title: {contracts[0]['title']}")

Dataset Loaded Successfully
Total Contracts: 510
Sample contract title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT


---
## Section 3 — Label Engineering (All 41 Labels)

Automatically extract ALL unique clause labels from the CUAD dataset.
Each QA entry has an `id` field formatted as: `contractname__LabelName`

In [3]:
# ==========================================
# AUTO-EXTRACT ALL 41 LABELS
# ==========================================

all_labels_set = set()
label_raw_counts = Counter()  # total QA entries per label
label_positive_counts = Counter()  # answerable (positive) entries per label

for contract in contracts:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            label = qa["id"].split("__")[-1]
            all_labels_set.add(label)
            label_raw_counts[label] += 1
            if not qa["is_impossible"] and qa.get("answers"):
                label_positive_counts[label] += 1

# Sort alphabetically for deterministic ordering
all_labels = sorted(all_labels_set)

# Create mappings
label_to_index = {label: idx for idx, label in enumerate(all_labels)}
index_to_label = {idx: label for idx, label in enumerate(all_labels)}

print(f"Total unique labels: {len(all_labels)}")
print(f"\nAll {len(all_labels)} labels (sorted):")
print("-" * 50)
for i, label in enumerate(all_labels):
    pos = label_positive_counts[label]
    total = label_raw_counts[label]
    neg = total - pos
    print(f"  {i+1:2d}. {label:<45s} | pos={pos:4d} | neg={neg:4d} | total={total}")

print(f"\nTotal QA entries: {sum(label_raw_counts.values())}")
print(f"Total positive:   {sum(label_positive_counts.values())}")
print(f"Total negative:   {sum(label_raw_counts.values()) - sum(label_positive_counts.values())}")

# Identify rare labels (fewer than 20 positive examples)
rare_threshold = 20
rare_labels = [l for l in all_labels if label_positive_counts[l] < rare_threshold]
print(f"\nRare labels (< {rare_threshold} positives): {len(rare_labels)}")
for rl in rare_labels:
    print(f"  - {rl}: {label_positive_counts[rl]} positives")

Total unique labels: 41

All 41 labels (sorted):
--------------------------------------------------
   1. Affiliate License-Licensee                    | pos=  59 | neg= 451 | total=510
   2. Affiliate License-Licensor                    | pos=  23 | neg= 487 | total=510
   3. Agreement Date                                | pos= 470 | neg=  40 | total=510
   4. Anti-Assignment                               | pos= 374 | neg= 136 | total=510
   5. Audit Rights                                  | pos= 214 | neg= 296 | total=510
   6. Cap On Liability                              | pos= 275 | neg= 235 | total=510
   7. Change Of Control                             | pos= 121 | neg= 389 | total=510
   8. Competitive Restriction Exception             | pos=  76 | neg= 434 | total=510
   9. Covenant Not To Sue                           | pos= 100 | neg= 410 | total=510
  10. Document Name                                 | pos= 510 | neg=   0 | total=510
  11. Effective Date                    

---
## Section 4 — Multi-Label Classification Theory

### Why Multi-Label (not Multi-Class)?

In **multi-class** classification, each input belongs to exactly ONE class.
A softmax activation produces a probability distribution over mutually exclusive classes.

In **multi-label** classification, each input can belong to **MULTIPLE classes simultaneously**.
A legal contract clause can be relevant to multiple categories at once — for example,
a clause might discuss both "Cap On Liability" AND "Uncapped Liability" in the same paragraph.

### Why Sigmoid Activation (not Softmax)?

- **Softmax** forces all class probabilities to sum to 1.0, creating competition between labels.
  If one label has high probability, others are pushed down. This is WRONG for multi-label.
- **Sigmoid** treats each label as an independent binary decision.
  Each output node independently produces a probability in [0, 1].
  Multiple labels can have high probability simultaneously.

### Why BCEWithLogitsLoss (not CrossEntropyLoss)?

- **CrossEntropyLoss** combines softmax + negative log likelihood — designed for mutually exclusive classes.
- **BCEWithLogitsLoss** combines sigmoid + binary cross-entropy — applies independent binary loss per label.
  This is the correct loss for multi-label classification.

### Dataset Format

Each row contains a text snippet and a 41-dimensional binary vector:
```
| text | Affiliate License-Licensee | Affiliate License-Licensor | ... | Warranty Duration |
| snippet | 0 | 1 | ... | 0 |
```

---
## Section 5 — Reusable Extraction Functions

In [4]:
def extract_positive_text(context, answers):
    """Extract the answer span with surrounding context for a POSITIVE example.

    Uses the longest answer span for best coverage, then expands a window
    of CONTEXT_WINDOW characters on each side for surrounding context.
    """
    # Use the longest answer span for best coverage
    best = max(answers, key=lambda a: len(a["text"]))
    start = best["answer_start"]
    end = start + len(best["text"])

    # Expand window to include surrounding context
    window_start = max(0, start - CONTEXT_WINDOW)
    window_end = min(len(context), end + CONTEXT_WINDOW)

    snippet = context[window_start:window_end].strip()
    # Clean up excessive whitespace
    snippet = " ".join(snippet.split())
    return snippet


def extract_negative_text(context):
    """Extract a random snippet from the contract for a NEGATIVE example.

    Selects a random position in the cleaned contract text and extracts
    NEGATIVE_SNIPPET_LEN characters.
    """
    cleaned = " ".join(context.split())
    if len(cleaned) <= NEGATIVE_SNIPPET_LEN:
        return cleaned
    start = random.randint(0, len(cleaned) - NEGATIVE_SNIPPET_LEN)
    snippet = cleaned[start:start + NEGATIVE_SNIPPET_LEN].strip()
    return snippet


def normalize_text(text):
    """Normalize whitespace and strip text for consistent deduplication."""
    return " ".join(text.split()).strip()


def compute_text_hash(text):
    """Compute MD5 hash for efficient text deduplication."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def extract_contract_id(contract):
    """Extract the contract title as a stable identifier for provenance tracking."""
    return contract.get("title", "unknown")


print("Extraction functions defined successfully.")
print(f"  CONTEXT_WINDOW:      {CONTEXT_WINDOW} chars")
print(f"  NEGATIVE_SNIPPET_LEN: {NEGATIVE_SNIPPET_LEN} chars")

Extraction functions defined successfully.
  CONTEXT_WINDOW:      250 chars
  NEGATIVE_SNIPPET_LEN: 500 chars


---
## Section 6 — Multi-Label Dataset Construction

**Strategy (Two-Pass Aggregation)**:

1. **Pass 1**: Collect all `(snippet, label, target, contract_id)` tuples
2. **Pass 2**: Aggregate by unique snippet → create one multi-label vector per snippet

This avoids duplicating the same text 41 times.

> **Note on negative snippet noise**: Random negative snippets may unintentionally
> contain valid clause language not annotated for that QA pair. This is a known CUAD
> limitation, acceptable for now. The model learns strong signal from positives.

In [5]:
# ==========================================
# PASS 1: Collect all (snippet, label, target, contract_id) tuples
# ==========================================

print("Pass 1: Extracting snippets from all contracts...")

raw_records = []  # list of (text, label, target, contract_id)

for contract in contracts:
    contract_id = extract_contract_id(contract)

    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]

        for qa in paragraph["qas"]:
            label = qa["id"].split("__")[-1]

            if not qa["is_impossible"] and qa.get("answers"):
                # POSITIVE: extract the specific clause with context
                snippet = extract_positive_text(context, qa["answers"])
                target = 1
            else:
                # NEGATIVE: extract a random excerpt from the contract
                snippet = extract_negative_text(context)
                target = 0

            # Normalize for consistent deduplication
            snippet = normalize_text(snippet)

            if snippet:  # skip empty snippets
                raw_records.append((snippet, label, target, contract_id))

print(f"  Total raw records collected: {len(raw_records)}")
print(f"  Unique contracts:            {len(set(r[3] for r in raw_records))}")

Pass 1: Extracting snippets from all contracts...


  Total raw records collected: 20910
  Unique contracts:            510


In [6]:
# ==========================================
# PASS 2: Aggregate into multi-label vectors
# ==========================================

print("Pass 2: Aggregating into multi-label vectors...")

# Group by (text_hash) -> accumulate labels and contract_id
snippet_data = defaultdict(lambda: {
    "text": "",
    "contract_id": "",
    "labels": np.zeros(len(all_labels), dtype=int)
})

for text, label, target, contract_id in raw_records:
    text_hash = compute_text_hash(text)

    if not snippet_data[text_hash]["text"]:
        snippet_data[text_hash]["text"] = text
        snippet_data[text_hash]["contract_id"] = contract_id

    if target == 1:
        label_idx = label_to_index[label]
        # Logical OR: if ANY occurrence is positive, keep it positive
        snippet_data[text_hash]["labels"][label_idx] = 1

print(f"  Unique snippets after aggregation: {len(snippet_data)}")

# Count multi-label statistics
label_counts_per_snippet = [int(v["labels"].sum()) for v in snippet_data.values()]
multi_label_counter = Counter(label_counts_per_snippet)
print(f"\n  Multi-label distribution:")
for n_labels in sorted(multi_label_counter.keys()):
    count = multi_label_counter[n_labels]
    print(f"    {n_labels} active labels: {count} snippets ({100*count/len(snippet_data):.1f}%)")

Pass 2: Aggregating into multi-label vectors...
  Unique snippets after aggregation: 20104

  Multi-label distribution:
    0 active labels: 14178 snippets (70.5%)
    1 active labels: 5251 snippets (26.1%)
    2 active labels: 606 snippets (3.0%)
    3 active labels: 56 snippets (0.3%)
    4 active labels: 12 snippets (0.1%)
    5 active labels: 1 snippets (0.0%)


---
## Section 7 — DataFrame Construction

In [7]:
# ==========================================
# BUILD FINAL DATAFRAME
# ==========================================

print("Building final DataFrame...")

rows = []
for text_hash, info in snippet_data.items():
    row = {"text": info["text"], "contract_id": info["contract_id"]}
    for i, label in enumerate(all_labels):
        row[label] = int(info["labels"][i])
    rows.append(row)

df = pd.DataFrame(rows)

# Ensure column order: text, contract_id, then all labels alphabetically
column_order = ["text", "contract_id"] + all_labels
df = df[column_order]

# Ensure all label columns are int type
for label in all_labels:
    df[label] = df[label].astype(int)

# Fill any NaN with 0 (safety net)
df = df.fillna(0)

# Clean whitespace in text column
df["text"] = df["text"].str.strip()

# Drop any exact duplicate rows
initial_len = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"  Dropped {initial_len - len(df)} duplicate rows")

print(f"\nFinal dataset shape: {df.shape}")
print(f"  Rows:    {df.shape[0]}")
print(f"  Columns: {df.shape[1]} (text + contract_id + {len(all_labels)} labels)")

print("\nSample rows (first 5):")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 10)
display(df.head())
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_columns")

Building final DataFrame...


  Dropped 0 duplicate rows

Final dataset shape: (20104, 43)
  Rows:    20104
  Columns: 43 (text + contract_id + 41 labels)

Sample rows (first 5):


,text,contract_id,Affiliate License-Licensee,Affiliate License-Licensor,Agreement Date,...,Third Party Beneficiary,Uncapped Liability,Unlimited/All-You-Can-Eat-License,Volume Restriction,Warranty Duration
0,"EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGREEMENT (the ""Agreemen...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,0,...,0,0,0,0,0
1,"mailed, two (2) business days after the date of deposit in the United States...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,0,...,0,0,0,0,0
2,"DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGREEMENT (the ""Agreement"") is made b...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,1,...,0,0,0,0,0
3,"agreement reflecting the terms and conditions of this Agreement, may be exec...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,0,...,0,0,0,0,0
4,"nt shall be ten (10) years (the ""Term"") which shall commence on the date upo...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,0,...,0,0,0,0,0


---
## Section 8 — Dataset Validation

In [8]:
# ==========================================
# COMPREHENSIVE VALIDATION
# ==========================================

print("=" * 60)
print("DATASET VALIDATION REPORT")
print("=" * 60)

# --- Shape ---
print(f"\n1. SHAPE")
print(f"   Rows:    {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")
expected_cols = 2 + len(all_labels)  # text + contract_id + labels
assert df.shape[1] == expected_cols, f"Expected {expected_cols} columns, got {df.shape[1]}"
print(f"   ✓ Column count matches expected ({expected_cols})")

# --- Null Check ---
print(f"\n2. NULL CHECK")
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f"   Total nulls: {total_nulls}")
if total_nulls > 0:
    print("   ⚠ Columns with nulls:")
    for col in null_counts[null_counts > 0].index:
        print(f"     - {col}: {null_counts[col]}")
else:
    print("   ✓ No null values found")

# --- Duplicate Check ---
print(f"\n3. DUPLICATE CHECK")
dup_count = df.duplicated().sum()
print(f"   Duplicate rows: {dup_count}")
text_dup_count = df["text"].duplicated().sum()
print(f"   Duplicate texts: {text_dup_count}")
print(f"   ✓ Deduplication {'complete' if dup_count == 0 else 'INCOMPLETE'}")

# --- Text Length Stats ---
print(f"\n4. TEXT LENGTH STATISTICS (characters)")
text_lengths = df["text"].str.len()
print(f"   Min:    {text_lengths.min()}")
print(f"   Max:    {text_lengths.max()}")
print(f"   Mean:   {text_lengths.mean():.1f}")
print(f"   Median: {text_lengths.median():.1f}")
print(f"   Std:    {text_lengths.std():.1f}")

# --- Label Frequency Distribution ---
print(f"\n5. LABEL FREQUENCY DISTRIBUTION (positive counts)")
label_freq = df[all_labels].sum().sort_values(ascending=False)
print(f"   {'Label':<45s} | Pos Count | Pos Rate")
print("   " + "-" * 75)
for label, count in label_freq.items():
    rate = 100 * count / len(df)
    print(f"   {label:<45s} | {int(count):>9d} | {rate:>6.2f}%")

# --- Multi-label Statistics ---
print(f"\n6. MULTI-LABEL STATISTICS")
active_per_row = df[all_labels].sum(axis=1)
print(f"   Avg active labels per snippet: {active_per_row.mean():.2f}")
print(f"   Max active labels per snippet: {int(active_per_row.max())}")
print(f"   Distribution:")
for n in sorted(active_per_row.unique()):
    c = (active_per_row == n).sum()
    print(f"     {int(n)} labels: {c} snippets ({100*c/len(df):.1f}%)")

# --- Imbalance Analysis ---
print(f"\n7. IMBALANCE ANALYSIS")
rare_threshold_pct = 5.0
rare = [(l, int(label_freq[l]), 100*label_freq[l]/len(df))
        for l in all_labels if 100*label_freq[l]/len(df) < rare_threshold_pct]
common = [(l, int(label_freq[l]), 100*label_freq[l]/len(df))
          for l in all_labels if 100*label_freq[l]/len(df) >= rare_threshold_pct]
print(f"   Rare labels (<{rare_threshold_pct}% positive): {len(rare)}")
for l, c, r in rare[:10]:
    print(f"     - {l}: {c} ({r:.2f}%)")
if len(rare) > 10:
    print(f"     ... and {len(rare)-10} more")
print(f"   Common labels (>={rare_threshold_pct}% positive): {len(common)}")

# --- Contract Coverage ---
print(f"\n8. CONTRACT COVERAGE")
unique_contracts = df["contract_id"].nunique()
print(f"   Unique contracts: {unique_contracts}")
snippets_per_contract = df.groupby("contract_id").size()
print(f"   Snippets per contract:")
print(f"     Min:    {snippets_per_contract.min()}")
print(f"     Max:    {snippets_per_contract.max()}")
print(f"     Mean:   {snippets_per_contract.mean():.1f}")
print(f"     Median: {snippets_per_contract.median():.1f}")

print(f"\n{'=' * 60}")
print("VALIDATION COMPLETE")
print(f"{'=' * 60}")

DATASET VALIDATION REPORT

1. SHAPE
   Rows:    20104
   Columns: 43
   ✓ Column count matches expected (43)

2. NULL CHECK
   Total nulls: 0
   ✓ No null values found

3. DUPLICATE CHECK
   Duplicate rows: 0
   Duplicate texts: 0
   ✓ Deduplication complete

4. TEXT LENGTH STATISTICS (characters)
   Min:    140
   Max:    3615
   Mean:   576.2
   Median: 500.0
   Std:    218.7

5. LABEL FREQUENCY DISTRIBUTION (positive counts)
   Label                                         | Pos Count | Pos Rate
   ---------------------------------------------------------------------------
   Document Name                                 |       508 |   2.53%
   Parties                                       |       508 |   2.53%
   Agreement Date                                |       468 |   2.33%
   Governing Law                                 |       436 |   2.17%
   Expiration Date                               |       411 |   2.04%
   Effective Date                                |       387 |

---
## Section 9 — Visualization

In [9]:
# ==========================================
# BAR CHART: Positive count per label
# ==========================================

fig, ax = plt.subplots(figsize=(14, 10))
label_freq_sorted = df[all_labels].sum().sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(label_freq_sorted)))
label_freq_sorted.plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Positive Count", fontsize=12)
ax.set_title("Positive Examples per Label (All 41 CUAD Categories)", fontsize=14, fontweight="bold")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "label_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Label distribution chart saved.")

Label distribution chart saved.


C:\Users\chari\AppData\Local\Temp\ipykernel_27452\819614284.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ==========================================
# HISTOGRAM: Active labels per snippet
# ==========================================

fig, ax = plt.subplots(figsize=(10, 6))
active_per_row = df[all_labels].sum(axis=1)
active_per_row.hist(bins=range(0, int(active_per_row.max()) + 2), ax=ax,
                    color="#4C72B0", edgecolor="white", alpha=0.85)
ax.set_xlabel("Number of Active Labels", fontsize=12)
ax.set_ylabel("Number of Snippets", fontsize=12)
ax.set_title("Distribution of Active Labels per Snippet", fontsize=14, fontweight="bold")
ax.set_xticks(range(0, int(active_per_row.max()) + 1))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "multilabel_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Multi-label distribution chart saved.")

Multi-label distribution chart saved.


C:\Users\chari\AppData\Local\Temp\ipykernel_27452\4089638474.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 10 — Train/Test Leakage Prevention

> ⚠️ **CRITICAL for legal NLP evaluation.**
>
> Splitting by random rows would leak contract-specific writing style into both
> train and test sets, inflating metrics. The model would learn to recognize
> *document style* rather than *actual legal semantics*.
>
> **All downstream train/test splits MUST group by `contract_id`.**
> Never split individual snippets randomly across train and test.

In [11]:
def get_contract_level_split(df, test_size=0.2, seed=42):
    """Split dataset at the CONTRACT level to prevent data leakage.

    Ensures no contract appears in both train and test sets.
    This is critical for legal NLP to evaluate generalization
    to unseen contracts, not memorization of writing styles.

    Args:
        df: DataFrame with 'contract_id' column
        test_size: fraction of contracts for test set (default 0.2)
        seed: random seed for reproducibility

    Returns:
        train_df, test_df: DataFrames with no contract overlap
    """
    rng = np.random.RandomState(seed)

    # Get unique contract IDs
    unique_contracts = df["contract_id"].unique()
    rng.shuffle(unique_contracts)

    # Split contract IDs
    split_idx = int(len(unique_contracts) * (1 - test_size))
    train_contracts = set(unique_contracts[:split_idx])
    test_contracts = set(unique_contracts[split_idx:])

    # Split dataframe
    train_df = df[df["contract_id"].isin(train_contracts)].reset_index(drop=True)
    test_df = df[df["contract_id"].isin(test_contracts)].reset_index(drop=True)

    return train_df, test_df


# Demonstrate the split
train_df, test_df = get_contract_level_split(df, test_size=0.2, seed=RANDOM_SEED)

# Verify zero overlap
train_contracts = set(train_df["contract_id"].unique())
test_contracts = set(test_df["contract_id"].unique())
overlap = train_contracts & test_contracts

print("CONTRACT-LEVEL SPLIT RESULTS")
print("=" * 50)
print(f"  Train contracts: {len(train_contracts)}")
print(f"  Test contracts:  {len(test_contracts)}")
print(f"  Overlap:         {len(overlap)} {'✓ ZERO LEAKAGE' if len(overlap) == 0 else '⚠ LEAKAGE DETECTED!'}")
print(f"\n  Train snippets: {len(train_df)}")
print(f"  Test snippets:  {len(test_df)}")
print(f"  Train ratio:    {100*len(train_df)/len(df):.1f}%")
print(f"  Test ratio:     {100*len(test_df)/len(df):.1f}%")

# Label distribution per split
print(f"\n  Label distribution (top 10 by train count):")
train_label_counts = train_df[all_labels].sum().sort_values(ascending=False)
test_label_counts = test_df[all_labels].sum()
print(f"  {'Label':<40s} | Train | Test")
print("  " + "-" * 65)
for label in train_label_counts.head(10).index:
    print(f"  {label:<40s} | {int(train_label_counts[label]):>5d} | {int(test_label_counts[label]):>5d}")

CONTRACT-LEVEL SPLIT RESULTS
  Train contracts: 408
  Test contracts:  102
  Overlap:         0 ✓ ZERO LEAKAGE

  Train snippets: 16073
  Test snippets:  4031
  Train ratio:    79.9%
  Test ratio:     20.1%

  Label distribution (top 10 by train count):
  Label                                    | Train | Test
  -----------------------------------------------------------------
  Document Name                            |   406 |   102
  Parties                                  |   406 |   102
  Agreement Date                           |   374 |    94
  Governing Law                            |   345 |    91
  Expiration Date                          |   330 |    81
  Effective Date                           |   304 |    83
  Anti-Assignment                          |   299 |    75
  Cap On Liability                         |   211 |    64
  License Grant                            |   202 |    52
  Audit Rights                             |   167 |    47


---
## Section 11 — Save Outputs

In [12]:
# ==========================================
# SAVE CSV DATASET
# ==========================================

df.to_csv(CSV_OUTPUT, index=False)
print(f"✓ Dataset saved: {CSV_OUTPUT}")
print(f"  Size: {os.path.getsize(CSV_OUTPUT) / (1024*1024):.2f} MB")

✓ Dataset saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\multi_label_clause_dataset.csv
  Size: 14.10 MB


In [13]:
# ==========================================
# SAVE LABEL MAPPING
# ==========================================

label_mapping = {
    "all_labels": all_labels,
    "label_to_index": label_to_index,
    "index_to_label": {str(k): v for k, v in index_to_label.items()},
    "total_labels": len(all_labels)
}

with open(LABEL_MAP_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

print(f"✓ Label mapping saved: {LABEL_MAP_OUTPUT}")

✓ Label mapping saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\label_mapping.json


In [14]:
# ==========================================
# SAVE DATASET STATISTICS
# ==========================================

active_per_row = df[all_labels].sum(axis=1)
label_freq = df[all_labels].sum()

dataset_statistics = {
    "dataset_shape": {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
    "num_labels": len(all_labels),
    "num_unique_contracts": int(df["contract_id"].nunique()),
    "text_length_stats": {
        "min": int(df["text"].str.len().min()),
        "max": int(df["text"].str.len().max()),
        "mean": round(float(df["text"].str.len().mean()), 1),
        "median": round(float(df["text"].str.len().median()), 1),
        "std": round(float(df["text"].str.len().std()), 1)
    },
    "label_positive_counts": {label: int(label_freq[label]) for label in all_labels},
    "multi_label_distribution": {
        str(int(k)): int(v) for k, v in
        sorted(Counter(active_per_row.astype(int).tolist()).items())
    },
    "avg_active_labels_per_snippet": round(float(active_per_row.mean()), 3),
    "contract_split_demo": {
        "train_contracts": len(train_contracts),
        "test_contracts": len(test_contracts),
        "train_snippets": len(train_df),
        "test_snippets": len(test_df),
        "leakage": len(overlap) > 0
    },
    "null_count": int(df.isnull().sum().sum()),
    "duplicate_count": int(df.duplicated().sum())
}

with open(STATS_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(dataset_statistics, f, indent=2, ensure_ascii=False)

print(f"✓ Statistics saved: {STATS_OUTPUT}")

✓ Statistics saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\dataset_statistics.json


---
## Section 12 — Final Summary

> **IMPORTANT REMINDER**: When fine-tuning Legal-BERT on this dataset,
> always split using `get_contract_level_split()` — NEVER random row splitting.
> This prevents data leakage from contract writing style.

In [15]:
print("=" * 60)
print("MULTI-LABEL CLAUSE DATASET — FINAL SUMMARY")
print("=" * 60)
print(f"\n  Dataset rows:     {df.shape[0]}")
print(f"  Dataset columns:  {df.shape[1]}")
print(f"  Total labels:     {len(all_labels)}")
print(f"  Unique contracts: {df['contract_id'].nunique()}")
print(f"  Avg labels/row:   {active_per_row.mean():.2f}")
print(f"  Nulls:            {df.isnull().sum().sum()}")
print(f"  Duplicates:       {df.duplicated().sum()}")
print(f"\nFiles saved:")
print(f"  1. {CSV_OUTPUT}")
print(f"  2. {LABEL_MAP_OUTPUT}")
print(f"  3. {STATS_OUTPUT}")
print(f"\n⚔️  Train/test splits MUST use contract_id grouping!")
print(f"    Use: get_contract_level_split(df, test_size=0.2)")
print(f"\n✓ Dataset ready for multi-label Legal-BERT fine-tuning")
print(f"  - 41-label sigmoid classification")
print(f"  - BCEWithLogitsLoss")
print(f"  - HuggingFace Trainer compatible")
print("=" * 60)

MULTI-LABEL CLAUSE DATASET — FINAL SUMMARY

  Dataset rows:     20104
  Dataset columns:  43
  Total labels:     41
  Unique contracts: 510
  Avg labels/row:   0.33
  Nulls:            0
  Duplicates:       0

Files saved:
  1. C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\multi_label_clause_dataset.csv
  2. C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\label_mapping.json
  3. C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\dataset_statistics.json

⚔️  Train/test splits MUST use contract_id grouping!
    Use: get_contract_level_split(df, test_size=0.2)

✓ Dataset ready for multi-label Legal-BERT fine-tuning
  - 41-label sigmoid classification
  - BCEWithLogitsLoss
  - HuggingFace Trainer compatible
